# Conhecendo o Spark: o que é e por que o time escolheu

O time escolheu o Spark porque o projeto da Arena Analytics vai crescer. Este notebook reúne os conceitos que
explicam essa escolha.

## 1. O que é processar dados

**Processar dados** é aplicar uma sequência de passos a dados brutos até chegar a um resultado útil. Exemplo do
projeto, gols por Copa:

1. **Ler** cada partida do arquivo
2. **Calcular** os gols da partida (gols do mandante + gols do visitante)
3. **Agrupar** as partidas pelo ano da Copa
4. **Somar** os gols de cada grupo

Em SQL: `SELECT ano, SUM(gols) FROM partidas GROUP BY ano`.

O mesmo cálculo com o **pandas**, uma biblioteca do Python para manipular dados em tabelas:

In [1]:
import pandas as pd

partidas = pd.read_csv("../../data/raw/WorldCupMatches.csv")
partidas = partidas.dropna(subset=["Year"])          # o arquivo tem linhas vazias no fim
partidas = partidas.astype({"Year": "int", "Home Team Goals": "int", "Away Team Goals": "int"})

partidas["gols"] = partidas["Home Team Goals"] + partidas["Away Team Goals"]
gols_por_copa = partidas.groupby("Year")["gols"].sum()

gols_por_copa.head()

Year
1930     70
1934     70
1938     92
1950     88
1954    172
Name: gols, dtype: int64

O pandas carrega o arquivo inteiro na **memória** e faz as contas numa máquina só.

## 2. O limite de uma máquina só

- **Memória (RAM):** onde o programa guarda o que está usando. Um notebook comum tem de 8 a 32 GB.
- **Processador (CPU):** quem faz as contas. Tem alguns **núcleos**, e cada núcleo executa uma coisa por vez.
- **Disco e rede:** a velocidade com que o dado chega até o processador.

O volume da Arena Analytics vai crescer: o feed de eventos tem mais de mil lances por partida nas Copas recentes, e dados de
rastreamento (a posição dos jogadores várias vezes por segundo) chegariam a milhões de linhas por partida. Com esse
volume, o dado não cabe na memória de uma máquina, ou o cálculo leva horas.

| Estratégia | Como funciona | Problema |
|---|---|---|
| **Escalar verticalmente** | Trocar por um computador maior (mais memória, mais núcleos) | Tem teto e fica caro muito rápido |
| **Escalar horizontalmente** | Usar vários computadores trabalhando juntos | É preciso coordenar o trabalho entre eles |

O Spark escala horizontalmente.

## 3. Processamento distribuído

**Processamento distribuído** é dividir o dado em pedaços, entregar cada pedaço a um computador diferente,
processar todos ao mesmo tempo e depois juntar os resultados.

```
                     arquivo de partidas
               /              |              \
         pedaço 1         pedaço 2         pedaço 3
       computador A     computador B     computador C

       1930: 25 gols    1930: 21 gols    1930: 24 gols     <- cada um soma só o seu pedaço
       1950: 30 gols    1950: 31 gols    1950: 27 gols        (resultado parcial)
       ...              ...              ...
               \              |              /
                juntar os parciais de cada Copa
                1930: 25 + 21 + 24 = 70 gols
                1950: 30 + 31 + 27 = 88 gols
```

Duas fases:

1. **Calcular em paralelo:** cada computador soma o seu pedaço, sem falar com os outros.
2. **Juntar:** os parciais de cada Copa se encontram num lugar só. Os computadores trocam dados pela rede, e isso
   tem custo.

## 4. Cluster

Um **cluster** é um grupo de computadores ligados em rede que trabalham como se fossem um só. Cada computador é um
**nó**. Em geral, os clusters ficam na nuvem (AWS, Azure, Google Cloud) ou no data center da empresa.

Coordenar um cluster na mão exigiria decidir quem fica com qual pedaço, como enviar o código, como juntar os
parciais e o que fazer quando uma máquina falha.

## 5. O que é o Apache Spark

Um **motor de processamento distribuído**, de código aberto. Você escreve a lógica uma vez, como se o dado fosse uma
tabela só, e o Spark:

- divide o dado em pedaços, chamados **partições**;
- distribui o trabalho entre os computadores do cluster;
- executa os pedaços em paralelo;
- junta os resultados;
- refaz automaticamente o pedaço de um computador que falhou.

Nasceu em 2009 na Universidade da Califórnia em Berkeley e é mantido pela Apache Software Foundation. Funciona com
Scala, Java, SQL e Python; o **PySpark** é a versão em Python. Sempre que possível, mantém os resultados
intermediários na memória, em vez de gravá-los em disco a cada passo.

## 6. Quando o Spark faz sentido

Comparando com o pandas, usado no começo do notebook:

| | pandas | Spark |
|---|---|---|
| Onde roda | numa máquina só, com o dado inteiro na memória | em várias máquinas (ou nos núcleos de uma), com o dado dividido em pedaços |
| Bom para | dado que cabe na memória de uma máquina e vai continuar cabendo | dado que não cabe numa máquina, ou que vai crescer até não caber |
| Custo de começar | quase nenhum: importa e calcula | alguns segundos para ligar e tempo coordenando as partes |

A pergunta que decide: **o dado cabe numa máquina hoje, e vai continuar cabendo?**

Tamanho dos arquivos de histórico da Arena Analytics hoje:

In [2]:
# liste os CSVs de ../../data/raw com o tamanho
from pathlib import Path

for arquivo in sorted(Path("../../data/raw").glob("*.csv")):
    print(f"{arquivo.name:<26} {arquivo.stat().st_size / 1024:>8.1f} KB")

WorldCupMatches.csv           163.1 KB
WorldCupPlayers.csv          1699.3 KB
WorldCups.csv                   1.6 KB


Hoje são poucos KB. Mas a cada Copa entram mais jogos (104 partidas por edição a partir de 2026), e o feed de
eventos multiplica o volume por centenas. Com o Spark, o pipeline é escrito uma vez e continua funcionando quando o
dado crescer.

## 7. Spark no seu computador: o modo local

No **modo local**, o Spark roda inteiro numa máquina só e usa os **núcleos** do processador como se fossem os
computadores do cluster. É o que a sessão pede com `master("local[*]")`.

```
num cluster (produção)                        no seu computador (modo local)
+-----------+ +-----------+ +-----------+     +------------------------------------------+
| máquina A | | máquina B | | máquina C |     | seu computador                           |
| pedaço 1  | | pedaço 2  | | pedaço 3  |     | núcleo 1   núcleo 2   núcleo 3   ...     |
+-----------+ +-----------+ +-----------+     | pedaço 1   pedaço 2   pedaço 3           |
                                              +------------------------------------------+
```

1. **O código é o mesmo.** Entre o modo local e um cluster, muda a configuração da sessão, não a lógica.
2. **No modo local, o limite é o do seu computador.** É onde o time desenvolve e testa o pipeline.
3. **Em produção, o cluster.** O pipeline roda num cluster na nuvem (por exemplo, Databricks, Amazon EMR ou Google
   Dataproc), com o mesmo código.

Núcleos disponíveis no container:

In [3]:
import os
os.cpu_count()

12

## Por que o time escolheu o Spark

- O histórico de hoje é pequeno, mas o feed de eventos e as próximas Copas vão multiplicar o volume.
- Com o Spark, o pipeline é escrito uma vez: roda no modo local durante o desenvolvimento e num cluster quando o
  volume pedir.
- Desenvolver com o dado de hoje é rápido: cada célula roda em segundos.